# Phase 3 Part 1: A Look at the Datasets #

In [1]:
import pandas as pd 

In [46]:
bandcamp_sales = pd.read_csv("data/1000000-bandcamp-sales.csv")
spotify_artists = pd.read_csv("data/spotify_artists.csv")
spotify_tracks = pd.read_csv("data/spotify-track-features.csv")

# The Datasets #

**1000000-bandcamp-sales:** https://www.kaggle.com/datasets/mathurinache/1000000-bandcamp-sales\
This dataset contains 1,000,000 individual Bandcamp sales transactions between Sept 9, 2020 - Oct 2, 2020. This is relevant to specifically question three of our proposal where we try to understand whether higher Spotify popularity and Bandcamp sales are associated within the specified timeframe. 

**spotify-track-features:** https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset\
This dataset contains Spotify’s track data, including track/artist meta-data, popularity scores, genres, and audio features. This is relevant to specifically question two of our proposal where we investigate the relationship between audio features such as danceability, tempo, and energy are associated with track popularity on Spotify,

**spotify-artists:** https://developer.spotify.com/documentation/web-api\
This dataset scraped from Spotify's web API via their developer dashboard contains indie and electronic artists' features such as release date frequency of albums, follower counts, and popularity scores. This is relevant to specifically question one of our proposal where we investigate the predictive abilities of Spotify artist attributes on popularity.

Our overarching research question that we aim to answer through these three smaller investigations is as follows:
### *What factors affect the success of indie and electronic musicians across different streaming platforms?* ###

# Data Cleaning #

## 1000000-bandcamp-sales ##

In [37]:
#bandcamp_sales.head()
bandcamp_sales.columns


Index(['_id', 'art_url', 'item_type', 'utc_date', 'country_code',
       'track_album_slug_text', 'country', 'slug_type', 'amount_paid_fmt',
       'item_price', 'item_description', 'art_id', 'url', 'amount_paid',
       'releases', 'artist_name', 'currency', 'album_title', 'amount_paid_usd',
       'package_image_id', 'amount_over_fmt', 'item_slug', 'addl_count'],
      dtype='str')

We first clean by removing columns that are unnecessary to our data analysis. These include:
- `track_album_slug_text`
- `item_type`
- `utc_date`
- `slug_type`
- `art_url`
- `item_description`
- `url`
- `album_title`
- `releases`
- `item_slug`
- `country`
- `country_code`
- `addl_count`
- `amount_paid_fmt`
- `currency`
- `package_image_id`

This leaves us with 6 columns:
- `_id`
- `item_type` * a for digital albums, t for digital tracks, p for physical items
- `item_price` * note that bandcamp has "pay what you want" option so some prices may be 0 while paid amount =/= 0
- `amount_paid`
- `currency`
- `amount_paid_usd`

In [49]:
bandcamp_sales = bandcamp_sales.drop(columns=[
    'track_album_slug_text','utc_date','art_url',
    'item_description','url','album_title','releases',
    'item_slug','country','country_code','addl_count',
    'amount_paid_fmt', 'slug_type', 'package_image_id', 'art_id'
], errors='ignore')

In [50]:
bandcamp_sales.columns
bandcamp_sales.isna().sum()

_id                 0
item_type           0
item_price          0
amount_paid         0
artist_name        10
currency            0
amount_paid_usd     0
dtype: int64

For further cleaning, we shoudl get rid of the na rows in `artist_name` (the only column with na values) as analysis cannot be done if we do not know who the artist even is. Thankfully, there are only 10 missing artists names so this should not impact analysis at all. 

In [51]:
bandcamp_sales = bandcamp_sales.dropna(subset=['artist_name'])
bandcamp_sales.isna().sum()
len(bandcamp_sales) #make sure 10 rows exactly were dropped from the total dataset

999990

## spotify-track-features ## 

In [58]:
spotify_tracks.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='str')

We first clean by removing columns that are unnecessary to our data analysis. These include:
- `Unnamed: 0`
- `album_name`

Due to department database quotas we should limit the dataset to fewer columns. Remove potential cases of multicollinearity and redundancies: 
- `mode` * 1 for major scale and 0 for minor scale
- `key`
- `loudness`
- `speechiness`
- `liveness`
- `instrumentalness`
- `energy`

This leaves us with 11 columns:
- `track_id`
- `artists` 
- `track_name`
- `popularity`
- `duration_ms`
- `explicit`
- `danceability`
- `acousticness`
- `tempo`
- `valence`
- `track_genre`

In [72]:
spotify_tracks = spotify_tracks.drop(columns=[
    'Unnamed: 0', 'album_name', 'mode', 'key', 'loudness', 'speechiness', 'liveness', 'instrumentalness', 'energy', 'time_signature'
], errors='ignore')
spotify_tracks.isna().sum()

track_id        0
artists         0
track_name      0
popularity      0
duration_ms     0
explicit        0
danceability    0
acousticness    0
valence         0
tempo           0
track_genre     0
dtype: int64

Similar to the previous dataset, just remove the row with na values because if we do not know the track name, it is useless to our analysis. 

In [ ]:
spotify_tracks = spotify_tracks.dropna(subset=['track_name'])

## spotify-artist ##

In [76]:
spotify_artists.columns

Index(['name', 'spotify_id', 'popularity', 'followers', 'genres',
       'total_releases', 'first_release', 'latest_release', 'years_active',
       'releases_per_year'],
      dtype='str')

I specifically scraped these features from the Spotify web API so no columns will be removed. The columns are: 
- `name`
- `spotify_id`
- `popularity`
- `followers`
- `genres`
- `total_releases`
- `first_release`
- `latest_release`
- `years_active`
- `releases_per_year`

In [96]:
spotify_artists = spotify_artists.dropna(subset=['first_release'])
spotify_artists.isna().sum()
len(spotify_artists)

3231

## Ensuring Consistent Column Names ##

Later on consistent column names will make joining schemas easier. Variables such as artist name, track name, and genre will be renamed across all datasets.

bandcamp:\
`_id` -> `bc_transaction_id`

tracks:\
`artists` -> `artist_name`\
`track_genre` -> `genre_type`

artists:\
`name` -> `artist_name`\
`genres` -> `genre_type`



In [90]:
bandcamp_sales.rename(columns={'_id': 'bc_transaction_id'}, inplace=True)
spotify_tracks.rename(columns={'artists': 'artist_name', 'track_genre': 'genre_type'}, inplace=True)
spotify_artists.rename(columns={'name': 'artist_name', 'genres':'genre_type'}, inplace = True)

bandcamp_sales.columns
spotify_tracks.columns
spotify_artists.columns

Index(['artist_name', 'spotify_id', 'popularity', 'followers', 'genre_type',
       'total_releases', 'first_release', 'latest_release', 'years_active',
       'releases_per_year'],
      dtype='str')

In [91]:
#save cleaned datasets
bandcamp_sales.to_csv('bandcamp_sales_cleaned.csv', index=False)
spotify_tracks.to_csv('spotify_tracks_cleaned.csv', index=False)
spotify_artists.to_csv('spotify_artists_cleaned.csv', index=False)

# Schema Design #
**Entities:** 

Track(**track_id**, track_name, artist_name, popularity, duration_ms, explicit, danceability, acousticness, valence, tempo, genre_type)
- artist_name is a foreign key referring to Artists entity attribute of the same name
- track_name, artist_name must be unique (avoid duplication of same song between the artist)

Artist(**artist_name**, spotify_id, popularity, followers, genre_type, total_releases, first_release, latest_release, years_active, releases_per_year)
- spotify_id must be unique

Sale(**bc_transaction_id**, artist_name, item_type, item_price, amount_paid, currency, amount_paid_usd)
- artist_name is a foreign key referring to Artists entity attribute of the same name
- numeric values price, paid, must be >=0
